## Task 3: Parts T16 in Registered Vehicles

Determine how many unique parts T16 ended up in vehicles registered in Adelshofen.

**Relevant tables:**
- `Bestandteile_Komponente_K2ST2.csv`: maps `ID_T16` to `ID_K2ST2`
- `Bestandteile_Komponente_K2LE2.csv`: maps `ID_T16` to `ID_K2LE2`
- `Bestandteile_Fahrzeuge_OEM1_Typ11.csv`, `OEM1_Typ12`, `OEM2_Typ21`, `OEM2_Typ22`: map `ID_Sitze` to `ID_Fahrzeug`
- `Zulassungen_alle_Fahrzeuge.csv`: maps `IDNummer` to `Gemeinden`

**Merge pipeline:** T16 parts are linked to seat components (K2ST2 or K2LE2), which are linked to vehicles via `ID_Sitze`, which are then matched to registrations via `ID_Fahrzeug` = `IDNummer`. All four vehicle files must be included.

To identify which component files contain T16 parts, all `Bestandteile_Komponente` files are checked for columns containing "T16".

In [4]:
import pandas as pd
import glob

for file in glob.glob("data/Komponente/Bestandteile_Komponente_*.csv"):
    cols = pd.read_csv(file, sep=";", engine="python", nrows=0).columns
    t16_cols = [c for c in cols if "T16" in c]
    if t16_cols:
        print(f"{file}: {t16_cols}")

data/Komponente\Bestandteile_Komponente_K2LE2.csv: ['ID_T16']
data/Komponente\Bestandteile_Komponente_K2ST2.csv: ['ID_T16']


T16 parts appear in two component files: `K2ST2` and `K2LE2`. Both need to be included in the merge pipeline. Only the required columns are loaded to reduce memory usage.

In [ ]:
import pandas as pd

# Load registration table
registration = pd.read_csv(
    "data/Zulassungen/Zulassungen_alle_Fahrzeuge.csv",
    sep=";",
    engine="python",
    usecols=["IDNummer", "Gemeinden"]
)
registration.columns = registration.columns.str.strip()

# Load T16 -> K2ST2 mapping
k2st2 = pd.read_csv(
    "data/Komponente/Bestandteile_Komponente_K2ST2.csv",
    sep=";",
    engine="python",
    usecols=["ID_T16", "ID_K2ST2"]
)

# Load T16 -> K2LE2 mapping
k2le2 = pd.read_csv(
    "data/Komponente/Bestandteile_Komponente_K2LE2.csv",
    sep=";",
    engine="python",
    usecols=["ID_T16", "ID_K2LE2"]
)

Each vehicle file is read in chunks and merged with both component tables (K2ST2, K2LE2) to find T16 parts. The results are then joined with the registration table to get the municipality.

In [ ]:
vehicle_files = [
    "data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv",
    "data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv",
    "data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ21.csv",
    "data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ22.csv"
]

result_chunks = []

for file in vehicle_files:
    print("Reading:", file)
    for chunk in pd.read_csv(
        file, sep=";", engine="python",
        chunksize=50000, usecols=["ID_Sitze", "ID_Fahrzeug"]
    ):
        # Merge with K2ST2 seat components
        merged_k2st2 = chunk.merge(k2st2, left_on="ID_Sitze", right_on="ID_K2ST2", how="inner")
        merged_k2st2 = merged_k2st2.merge(registration, left_on="ID_Fahrzeug", right_on="IDNummer", how="inner")
        result_chunks.append(merged_k2st2[["ID_T16", "Gemeinden"]])

        # Merge with K2LE2 seat components
        merged_k2le2 = chunk.merge(k2le2, left_on="ID_Sitze", right_on="ID_K2LE2", how="inner")
        merged_k2le2 = merged_k2le2.merge(registration, left_on="ID_Fahrzeug", right_on="IDNummer", how="inner")
        result_chunks.append(merged_k2le2[["ID_T16", "Gemeinden"]])

Reading: data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv
Reading: data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv
Reading: data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ21.csv
Reading: data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ22.csv


The results from all files are combined. Duplicates are removed and the data is filtered for Adelshofen.

In [ ]:
# Combine and deduplicate
result = pd.concat(result_chunks, ignore_index=True)
result = result.drop_duplicates()

# Filter for Adelshofen
adelshofen = result[result["Gemeinden"].str.upper() == "ADELSHOFEN"]
count = adelshofen["ID_T16"].nunique()
print("Number of unique T16 parts in Adelshofen:", count)

Number of unique T16 parts in Adelshofen: 48


**Result:** 48 unique T16 parts were found in vehicles registered in Adelshofen.